# 02 · Download validation data for the season dates

Independent dates to check the Sentinel-2 season dates of `04` against, from the **MeteoSwiss phenology network** (open data, no account): European larch *needle colouring 50 %* and *needle drop 50 %*, observed by eye at stations near the AOI. Colouring 50 % corresponds to the yellowing peak / midpoint, needle drop 50 % to the end of the season. The stations lie lower than most of the AOI, so `04` compares year-to-year anomalies, not absolute dates.

- **Settings:** `config.py` (`VALIDATION_RADIUS_KM`, `PHENO_PARAMS`, years)
- **Reads:** `cache/<AOI>_zones/zones.tif` (from `01_download_sentinel2`)
- **Writes** to `cache/validation/<AOI>/`:
  - `meteoswiss_larch.csv`: one row per station, year and phase
  - `meteoswiss_stations.csv`: the stations used
  - `station_map.png`, `aoi_map.png`: maps of the stations and of the AOI with the Swiss National Park
- **Next:** `04_phenology_metrics` (section 7 compares both methods with these data)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray
from matplotlib import patheffects
from matplotlib.colors import to_rgba
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pyproj import Geod, Transformer
from rasterio.features import shapes
from shapely.geometry import shape

import importlib
import config, phenology_utils
importlib.reload(config); importlib.reload(phenology_utils)  # pick up edits to config.py without a kernel restart
from config import *
from phenology_utils import collection_asset, download, stac_items

PHENO_COLLECTION = "ch.meteoschweiz.ogd-phenology"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

zones_tif = rioxarray.open_rasterio(CACHE_DIR / "zones.tif", masked=True)
x0, y0, x1, y1 = zones_tif.rio.bounds()
to_wgs84 = Transformer.from_crs(zones_tif.rio.crs, "EPSG:4326", always_xy=True)
AOI_LON, AOI_LAT = to_wgs84.transform((x0 + x1) / 2, (y0 + y1) / 2)
print(f"AOI: {AOI_NAME} | centre {AOI_LAT:.3f} N, {AOI_LON:.3f} E | writes to {VALIDATION_DIR}")

## MeteoSwiss phenology observations (European larch)

The Swiss Phenology Network (about 160 stations) records the date when 50 % of the larch needles at a site are coloured (`mlard94d`) or have fallen (`mlard95d`). Data: MeteoSwiss open data, collection `ch.meteoschweiz.ogd-phenology` (one CSV per station, one column per phase, dates as YYYYMMDD).

Stations within `VALIDATION_RADIUS_KM` of the AOI centre that observe one of `PHENO_PARAMS` are kept.

In [ ]:
def read_meta(name):
    path = download(collection_asset(PHENO_COLLECTION, name), VALIDATION_DIR / "meteoswiss" / name)
    return pd.read_csv(path, sep=";", encoding="cp1252")


stations = read_meta("ogd-phenology_meta_stations.csv").set_index("station_abbr")
inventory = read_meta("ogd-phenology_meta_datainventory.csv")
observing = inventory[inventory.parameter_shortname.isin(PHENO_PARAMS)].station_abbr.unique()
lat, lon = stations.station_coordinates_wgs84_lat, stations.station_coordinates_wgs84_lon
stations["distance_km"] = 6371 * np.hypot(np.radians(lat - AOI_LAT), np.radians(lon - AOI_LON) * np.cos(np.radians(AOI_LAT)))
near = stations.loc[observing].query("distance_km <= @VALIDATION_RADIUS_KM").sort_values("distance_km")
near = near[["station_name", "station_height_masl", "distance_km", "station_coordinates_wgs84_lat",
             "station_coordinates_wgs84_lon"]].rename(columns=lambda c: c.replace("station_", "").replace("coordinates_wgs84_", ""))
print(f"{len(near)} stations within {VALIDATION_RADIUS_KM} km observe larch needle colouring or drop")

# One CSV per station (STAC item id = lower-case abbreviation)
assets = {item["id"].upper(): next(iter(item["assets"].values()))["href"] for item in stac_items(PHENO_COLLECTION)}
rows = []
for abbr in near.index:
    path = download(assets[abbr], VALIDATION_DIR / "meteoswiss" / f"{abbr.lower()}_y.csv")
    data = pd.read_csv(path, sep=";", encoding="cp1252", dtype=str)
    year = pd.to_datetime(data.reference_timestamp, format="%d.%m.%Y %H:%M").dt.year
    for param, phase in PHENO_PARAMS.items():
        if param not in data:
            continue
        when = pd.to_datetime(data[param], format="%Y%m%d", errors="coerce")
        ok = when.notna()
        rows.append(pd.DataFrame({"station": abbr, "year": year[ok], "param": param, "phase": phase,
                                  "date": when[ok].dt.date, "doy": when[ok].dt.dayofyear}))
larch_obs = pd.concat(rows, ignore_index=True).join(near[["name", "height_masl", "distance_km"]], on="station")
larch_obs = larch_obs[larch_obs.year.between(FIRST_YEAR - 10, LAST_YEAR)]   # some years before for the station means
near = near.join(larch_obs[larch_obs.year >= FIRST_YEAR].pivot_table(index="station", columns="param", values="year",
                                                                    aggfunc="nunique").add_prefix("years_"))
near = near.dropna(how="all", subset=[c for c in near if c.startswith("years_")])   # no observations since FIRST_YEAR
display(near.round(1))

larch_obs.to_csv(VALIDATION_DIR / "meteoswiss_larch.csv", index=False)
near.to_csv(VALIDATION_DIR / "meteoswiss_stations.csv")

fig, axes = plt.subplots(1, len(PHENO_PARAMS), figsize=(12, 4), sharey=True)
for ax, (param, phase) in zip(axes, PHENO_PARAMS.items()):
    for abbr, g in larch_obs[(larch_obs.param == param) & (larch_obs.year >= FIRST_YEAR)].groupby("station"):
        ax.plot(g.year, g.doy, "o-", ms=4, label=f"{abbr} ({near.height_masl[abbr]:.0f} m)")
    ax.set_title(f"Larch {phase}")
    ax.set_xlabel("year")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("day of year")
axes[-1].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## Maps

Two maps for presentations, on the swisstopo national map (grey), saved at 300 dpi:

- `station_map.png`: the stations used and the AOI's larch pixels, with the search radius `VALIDATION_RADIUS_KM` around the AOI centre
- `aoi_map.png`: the AOI's larch pixels and the border of the Swiss National Park (BAFU, collection `ch.bafu.schutzgebiete-schweizerischer_nationalpark`)

Map tiles and the park border are cached in `cache/maps/`.

In [ ]:
MAP_TILES = "https://wmts.geo.admin.ch/1.0.0/ch.swisstopo.pixelkarte-grau/default/current/3857/{z}/{x}/{y}.jpeg"
MAP_CACHE = ROOT / "cache" / "maps"
ORIGIN = np.pi * 6378137   # m, half the circumference of the Web Mercator (EPSG:3857) sphere
LARCH_COLOR = "#d95f02"
LABEL_LEFT = {"SMO", "LLU"}   # stations labelled on the left, where labels would overlap


def basemap(ax, xmin, ymin, xmax, ymax, zoom):
    # swisstopo tiles covering a box in EPSG:3857 (cached), mosaicked and drawn faded
    size = 2 * ORIGIN / 2 ** zoom
    tx0, tx1 = int((xmin + ORIGIN) // size), int((xmax + ORIGIN) // size)
    ty0, ty1 = int((ORIGIN - ymax) // size), int((ORIGIN - ymin) // size)
    rows = []
    for y in range(ty0, ty1 + 1):
        row = []
        for x in range(tx0, tx1 + 1):
            path = MAP_CACHE / "tiles" / f"pixelkarte-grau_{zoom}_{x}_{y}.jpeg"
            if not path.exists():
                download(MAP_TILES.format(z=zoom, x=x, y=y), path)
            row.append(plt.imread(path))
        rows.append(np.hstack(row))
    img = np.vstack(rows)
    ax.imshow(img, extent=(tx0 * size - ORIGIN, (tx1 + 1) * size - ORIGIN, ORIGIN - (ty1 + 1) * size, ORIGIN - ty0 * size),
              cmap="gray" if img.ndim == 2 else None, interpolation="lanczos", alpha=0.6)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)


def map_furniture(ax, scale_km, credit):
    # Scale bar (Web Mercator stretches distances by 1 / cos(lat)), north arrow and credit; no axes
    (x0, x1), (y0, y1) = ax.get_xlim(), ax.get_ylim()
    bar = scale_km * 1000 / np.cos(np.radians(AOI_LAT))
    bx, by = x0 + 0.05 * (x1 - x0), y0 + 0.04 * (y1 - y0)
    ax.plot([bx, bx + bar], [by, by], color="0.1", lw=3, solid_capstyle="butt")
    ax.text(bx + bar / 2, by + 0.015 * (y1 - y0), f"{scale_km} km", ha="center", fontsize=9, color="0.1",
            path_effects=HALO)
    ax.annotate("N", xy=(0.95, 0.95), xytext=(0.95, 0.87), xycoords="axes fraction", ha="center", fontsize=11,
                fontweight="bold", arrowprops=dict(arrowstyle="-|>", color="0.1"), path_effects=HALO)
    ax.text(0.99, 0.005, credit, transform=ax.transAxes, ha="right", va="bottom", fontsize=7, color="0.3",
            path_effects=HALO)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")


HALO = [patheffects.withStroke(linewidth=3, foreground="white")]
to_mercator = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

# AOI: the larch pixels (band 2, elevation band, is set on every AOI pixel) as polygons
aoi_mask = zones_tif.isel(band=1).notnull().values
aoi = gpd.GeoSeries([shape(g) for g, _ in shapes(aoi_mask.astype("uint8"), mask=aoi_mask,
                                                 transform=zones_tif.rio.transform())], crs=zones_tif.rio.crs)
aoi_elev = float(zones_tif.isel(band=0).values[aoi_mask].mean())

# ---- Stations around the AOI, with the search radius (geodesic circle around the AOI centre)
lons, lats, _ = Geod(ellps="WGS84").fwd(np.full(361, AOI_LON), np.full(361, AOI_LAT), np.arange(361),
                                        np.full(361, VALIDATION_RADIUS_KM * 1000))
cx, cy = to_mercator.transform(lons, lats)
pad = 0.04 * (cx.max() - cx.min())

fig, ax = plt.subplots(figsize=(8, 8))
basemap(ax, cx.min() - pad, cy.min() - pad, cx.max() + pad, cy.max() + pad, zoom=12)
ax.plot(cx, cy, "--", color="0.2", lw=1.2)
aoi.to_crs("EPSG:3857").plot(ax=ax, color=LARCH_COLOR, edgecolor=LARCH_COLOR, lw=0.8)
sx, sy = to_mercator.transform(near.lon.values, near.lat.values)
ax.scatter(sx, sy, s=70, color="#1b4f8a", edgecolor="white", lw=1.2, zorder=3)
for x, y, row in zip(sx, sy, near.itertuples()):
    left = row.Index in LABEL_LEFT
    ax.annotate(f"{row.name}\n{row.height_masl:.0f} m", (x, y), xytext=(-6 if left else 6, 4),
                textcoords="offset points", ha="right" if left else "left", fontsize=9, fontweight="bold",
                color="0.1", path_effects=HALO, zorder=4)
map_furniture(ax, 10, "Map: © swisstopo | Stations: MeteoSwiss")
ax.legend(handles=[Patch(color=LARCH_COLOR, label=f"AOI larch stands (mean {aoi_elev:.0f} m)"),
                   Line2D([], [], ls="", marker="o", ms=8, color="#1b4f8a", mec="white",
                          label="MeteoSwiss larch phenology station"),
                   Line2D([], [], ls="--", color="0.2", label=f"search radius {VALIDATION_RADIUS_KM} km")],
          loc="upper left", fontsize=9, framealpha=0.9)
ax.set_title("Larch phenology stations around the AOI")
plt.tight_layout()
fig.savefig(VALIDATION_DIR / "station_map.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
SNP_COLLECTION = "ch.bafu.schutzgebiete-schweizerischer_nationalpark"
SNP_COLOR = "#1b7837"

# ---- The AOI and the Swiss National Park
snp_zip = MAP_CACHE / "nationalpark.shp.zip"
if not snp_zip.exists():
    snp_assets = stac_items(SNP_COLLECTION)[0]["assets"].values()
    download(next(a["href"] for a in snp_assets if a["href"].endswith(".shp.zip")), snp_zip)
snp = gpd.read_file(snp_zip)
inside = aoi.intersection(snp.to_crs(aoi.crs).union_all()).area.sum() / aoi.area.sum()   # share of the larch area

aoi_m, snp_m = aoi.to_crs("EPSG:3857"), snp.to_crs("EPSG:3857")
xmin, ymin, xmax, ymax = np.r_[np.minimum(aoi_m.total_bounds[:2], snp_m.total_bounds[:2]),
                       np.maximum(aoi_m.total_bounds[2:], snp_m.total_bounds[2:])]
pad = 0.06 * max(xmax - xmin, ymax - ymin)
fig, ax = plt.subplots(figsize=(8, 8 * (ymax - ymin + 2 * pad) / (xmax - xmin + 2 * pad)))
basemap(ax, xmin - pad, ymin - pad, xmax + pad, ymax + pad, zoom=13)
snp_m.plot(ax=ax, facecolor=SNP_COLOR, alpha=0.08)
snp_m.boundary.plot(ax=ax, color=SNP_COLOR, lw=2)
aoi_m.plot(ax=ax, color=LARCH_COLOR, edgecolor=LARCH_COLOR, lw=0.5)
map_furniture(ax, 5, "Map: © swisstopo | National Park: BAFU")
ax.legend(handles=[Patch(color=LARCH_COLOR, label=f"AOI larch stands ({inside:.0%} inside the park)"),
                   Patch(facecolor=to_rgba(SNP_COLOR, 0.08), edgecolor=SNP_COLOR, lw=2,
                         label="Swiss National Park")],
          loc="upper left", fontsize=9, framealpha=0.9)
ax.set_title("AOI larch stands and the Swiss National Park")
plt.tight_layout()
fig.savefig(VALIDATION_DIR / "aoi_map.png", dpi=300, bbox_inches="tight")
plt.show()